In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data"


In [2]:
identity = pd.read_csv(DATA_DIR / "train_identity.csv")
transactions = pd.read_csv(DATA_DIR / "train_transaction.csv")

In [3]:
print(identity.memory_usage(deep=True).sum())
print(identity.shape)

150097550
(144233, 41)


In [4]:
print(transactions.memory_usage(deep=True).sum())
print(transactions.shape)

2162238266
(590540, 394)


In [5]:
transactions.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
merged = pd.merge(transactions,identity, on="TransactionID", how='left')
merged.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [7]:
merged.shape

(590540, 434)

In [8]:
print(merged["DeviceType"].notna().mean(),
merged.memory_usage(deep=True).sum())

0.23844278118332374 2636093316


In [ ]:
int_cols = merged.select_dtypes(include="int64").columns
for col in int_cols:
    merged[col] = pd.to_numeric(merged[col], downcast="integer")
merged.head()
merged.memory_usage(deep=True).sum()

np.int64(2623691976)

In [9]:
print(merged.dtypes.value_counts())

float64    399
object      31
int64        4
Name: count, dtype: int64


In [10]:
float_cols = merged.select_dtypes(include="float64").columns
for col in float_cols:
    merged[col] = pd.to_numeric(merged[col], downcast="float")
merged.memory_usage(deep=True).sum()

np.int64(1693591476)

In [11]:
(merged.nunique()/len(merged)).sort_values()
object_cols = merged.select_dtypes(include="object").columns
obj_cols = []
for col in object_cols:
    if merged[col].memory_usage(deep=True) > merged[col].astype("category").memory_usage(deep=True):
        obj_cols.append(col)
for col in obj_cols:
    merged[col] = merged[col].astype("category")
merged.memory_usage(deep=True).sum()

np.int64(981720851)

In [13]:
merged.to_parquet()
PROCESSED_DIR = DATA_DIR / "processed"
merged.to_parquet(PROCESSED_DIR / "merged.parquet")

In [14]:
merged1 = pd.read_parquet(PROCESSED_DIR / "merged.parquet")
merged1.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
